# Getting Census Zipcode Income Breakdowns Data

Getting data from [https://data.census.gov/table?q=ZCTA5+11215&t=Income+and+Poverty&g=010XX00US]

for example looking for 2023 subject table for 11215 zip code

In [ ]:
import os
from dotenv import load_dotenv
import requests
load_dotenv()
census_api_key = os.environ.get("CENSUS_API_KEY")
import pandas as pd

In [ ]:

# random nyc zip code
zcta='11215'
# income breakdown from 2023 group
group = "S0101"

url = f"https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,group(S1901)&for=zip%20code%20tabulation%20area:{zcta}&key={census_api_key}"


response = requests.get(url)

In [ ]:
# Checking if the request was successful
data = None
if response.status_code == 200:
    data = response.json()
else:
    # Handling errors
    print(f"Error: {response.status_code}, {response.text}")

if data:
    data = pd.DataFrame(data)

data.columns = data.iloc[0]
data = data[1:]
# has lots of metrics we don't need
#also isnt human readable yet
data


,NAME,GEO_ID,NAME,S1901_C01_001E,S1901_C01_001EA,S1901_C01_001M,S1901_C01_001MA,S1901_C01_002E,S1901_C01_002EA,S1901_C01_002M,...,S1901_C04_014MA,S1901_C04_015E,S1901_C04_015EA,S1901_C04_015M,S1901_C04_015MA,S1901_C04_016E,S1901_C04_016EA,S1901_C04_016M,S1901_C04_016MA,zip code tabulation area
1,ZCTA5 11215,860Z200US11215,ZCTA5 11215,29747,None,1097,None,3.3,None,0.9,...,(X),-888888888,(X),-888888888,(X),28.4,None,-888888888.0,(X),11215


In [14]:


# Get variable definitions
vars_url = "https://api.census.gov/data/2023/acs/acs5/subject/variables.json"
vars_response = requests.get(vars_url).json()
variables = vars_response['variables']

# match labels for S1901 variables
s1901_labels = {
    var: info['label'] 
    for var, info in variables.items() 
    if var.startswith("S1901_C01_") and var.endswith("E")
}

s1901_labels = s1901_labels.items()
s1901_labels = pd.DataFrame(s1901_labels)
s1901_labels.rename(columns={0: 'code_label', 1: 'readable_label'}, inplace=True)
s1901_labels



,code_label,readable_label
0,S1901_C01_016E,Estimate!!Households!!PERCENT ALLOCATED!!Nonfa...
1,S1901_C01_015E,Estimate!!Households!!PERCENT ALLOCATED!!Famil...
2,S1901_C01_014E,Estimate!!Households!!PERCENT ALLOCATED!!House...
3,S1901_C01_013E,Estimate!!Households!!Mean income (dollars)
4,S1901_C01_012E,Estimate!!Households!!Median income (dollars)
5,S1901_C01_011E,"Estimate!!Households!!Total!!$200,000 or more"
6,S1901_C01_010E,"Estimate!!Households!!Total!!$150,000 to $199,999"
7,S1901_C01_009E,"Estimate!!Households!!Total!!$100,000 to $149,999"
8,S1901_C01_008E,"Estimate!!Households!!Total!!$75,000 to $99,999"
9,S1901_C01_007E,"Estimate!!Households!!Total!!$50,000 to $74,999"


In [ ]:
col_names = list(s1901_labels['code_label'])
columns_to_select = col_names + ['GEO_ID', 'zip code tabulation area']
filtered_data = data[columns_to_select]
filtered_data = filtered_data.rename(columns={"zip code tabulation area": "ZIP"})
filtered_data

,S1901_C01_016E,S1901_C01_015E,S1901_C01_014E,S1901_C01_013E,S1901_C01_012E,S1901_C01_011E,S1901_C01_010E,S1901_C01_009E,S1901_C01_008E,S1901_C01_007E,S1901_C01_006E,S1901_C01_005E,S1901_C01_004E,S1901_C01_003E,S1901_C01_002E,S1901_C01_001E,GEO_ID,ZIP
1,-888888888,-888888888,29.9,246090,180773,44.5,13.4,15.0,7.0,7.9,2.9,2.3,2.7,1.0,3.3,29747,860Z200US11215,11215
